# 04 Public Comps and Market Premium

The right output is not an average multiple. This notebook builds peer
groups, optional live market pulls, narrative premium scoring, and the
explicit market premium bridge from fundamental SOTP to IPO market value.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/spacex_ipo/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/spacex_ipo/data").exists()
)
USD_BN = 1_000_000_000
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
comps = pd.read_csv(DATA_DIR / "public_comps.csv")
comps.head()

In [ ]:
try:
    import yfinance as yf

    tickers = comps["ticker"].dropna().unique().tolist()
    rows = []
    for ticker in tickers:
        info = yf.Ticker(ticker).fast_info
        rows.append(
            {
                "ticker": ticker,
                "market_cap": getattr(info, "market_cap", np.nan),
                "last_price": getattr(info, "last_price", np.nan),
                "currency": getattr(info, "currency", None),
            }
        )
    live_market = pd.DataFrame(rows)
except Exception as exc:
    live_market = pd.DataFrame(
        [
            {
                "ticker": "DATA_UNAVAILABLE",
                "market_cap": np.nan,
                "last_price": np.nan,
                "currency": str(exc),
            }
        ]
    )
live_market.head()

In [ ]:
premium_inputs = pd.read_csv(DATA_DIR / "market_premiums.csv").set_index("case")
base_sotp = 1_350.0
premium_bridge = []
for case, row in premium_inputs.iterrows():
    result = market_premium_value(
        base_sotp,
        MarketPremiumInputs(
            musk_premium=row.musk_premium,
            ai_scarcity_premium=row.ai_scarcity_premium,
            ipo_scarcity_premium=row.ipo_scarcity_premium,
            strategic_asset_premium=row.strategic_asset_premium,
            governance_discount=row.governance_discount,
            execution_haircut=row.execution_haircut,
        ),
    )
    premium_bridge.append(
        {
            "case": case,
            "base_sotp_usd_bn": base_sotp,
            "net_premium": result.net_premium,
            "market_value_usd_bn": result.market_value,
        }
    )
pd.DataFrame(premium_bridge)

In [ ]:
comps.groupby("peer_group").agg(
    company_count=("company", "count"),
    avg_narrative_score=("narrative_premium_score", "mean"),
).sort_values("avg_narrative_score", ascending=False)